# 검수한 데이터를 학습 · 평가 · 시험으로 나누기
준비: `pairs_checked.csv` — 열: 본문, 리드, 판정(사용·수정·제외)
- **수정**한 쌍은 `리드` 칸을 고친 문장으로 바꿔 두세요.

In [11]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

실행 위치: 내 컴퓨터 (Windows)
GPU: NVIDIA GeForce RTX 5070 Laptop GPU, 8151 MiB
한글 글꼴: ['Malgun Gothic']


In [12]:
import pandas as pd
df = pd.read_csv('pairs_checked.csv')
print(df['판정'].value_counts())

판정
사용    313
제외     41
Name: count, dtype: int64


In [ ]:
df = df[df['판정'] != '제외'].copy()                           # 제외한 쌍 빼기
# df['instruction'] = '보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.'
df['instruction'] = '스포츠 리포트 본문을 읽고, 앵커가 읽을 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'
df['input'], df['output'] = df['본문'], df['리드']
test  = df.sample(5, random_state=42)                      # 마지막 비교용, 학습에 절대 안 씀
rest  = df.drop(test.index)
eval_ = rest.sample(frac=0.15, random_state=42)
train = rest.drop(eval_.index)
for d, name in [(train, 'train.jsonl'), (eval_, 'eval.jsonl'), (test, 'test.jsonl')]:
    d[['instruction', 'input', 'output']].to_json(name, orient='records', lines=True, force_ascii=False)
print(len(train), len(eval_), len(test)) 

262 46 5


## 다음 단계
1. 학습: Unsloth Studio 또는 `2_train_unsloth_gpu.ipynb`
2. 내보내기(GGUF) → `ollama create 내모델이름 -f Modelfile` (README 참고)
3. 비교: **`4_compare_results.ipynb`** 에서 세 모델 답을 받아 비교표 만들기